In [1]:
import tqdm

In [2]:
import re

In [3]:
import json

In [4]:
import pickle

In [5]:
from dotenv import load_dotenv

In [6]:
load_dotenv()

True

In [7]:
import pandas as pd

In [8]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [9]:
MODEL_ID = "google/gemma-4-E2B-it"

In [10]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [11]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [12]:
with open("titles_with_tags_dict.pkl", "rb") as f:
    titles_with_tags = pickle.load(f)

In [13]:
len(titles_with_tags)

1187

In [14]:
data = []

In [15]:
for title, tags in titles_with_tags.items():
    for tag in tags:
        data.append((title, tag))

In [16]:
data[0]

('Исследование приоритетов и механизмов реализации отраслевых (секторальных) политик в избранных странах БРИКС (членах и кандидатах), во взаимосвязи с повесткой международных доноров',
 'international_relations')

In [17]:
data[-1]

('Обзор Методологии в Области Регионоведения. Этап сбора данных 1.',
 'regional_studies')

In [18]:
df = pd.DataFrame(data=data, columns=["title_rus", "tag"])

In [19]:
df

,title_rus,tag
0,Исследование приоритетов и механизмов реализац...,international_relations
1,Исследование приоритетов и механизмов реализац...,political_economics
2,Исследование приоритетов и механизмов реализац...,policy_analysis
3,Исследование приоритетов и механизмов реализац...,geopolitics
4,Исследование приоритетов и механизмов реализац...,BRICS
...,...,...
6035,Актив Центра развития карьеры - профориентацио...,student_services
6036,Обзор Методологии в Области Регионоведения. Эт...,geography
6037,Обзор Методологии в Области Регионоведения. Эт...,methodology
6038,Обзор Методологии в Области Регионоведения. Эт...,data_collection


In [20]:
dfgr = df.groupby(by=["tag"]).agg({"title_rus": "nunique"}).reset_index()
dfgr.columns = ["tag", "count"]
dfgr = dfgr.sort_values(by=["count"], ascending=False).reset_index(drop=True)
dfgr.head(32)

,tag,count
0,international_relations,97
1,sociology,96
2,economics,93
3,linguistics,92
4,software_engineering,89
5,marketing,85
6,cultural_studies,76
7,education,74
8,history,73
9,media_studies,70


In [21]:
selected_tags = dfgr.head(32)["tag"].to_list()

In [22]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can create artificial user profiles based on student project titles in russian.
Please, analyze given list of student project titles in russian and create one possible user profile with own preferences in provided field of science.
Return stricly JSON output. Be brief.
For example.
Input data:
```json
{"machine_learning": ["Исследование методов решения задачи линейной регрессии", "Исследование методов решения задачи линейной классификации"]}
```
Your output:
```json
{"machine_learning_researcher": "A user, which is interested in machine learning and classic algorithms in particular"}
```
"""

In [23]:
messages = []

In [24]:
for tag in selected_tags:
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": json.dumps(
                    {
                        tag: list(df[df["tag"] == tag]["title_rus"].unique())
                    }, ensure_ascii=False
                ),
            },
        )
    )

In [25]:
messages[0][1]["content"]

'{"international_relations": ["Исследование приоритетов и механизмов реализации отраслевых (секторальных) политик в избранных странах БРИКС (членах и кандидатах), во взаимосвязи с повесткой международных доноров", "Сеть военно-политических союзов в Евразии: база данных", "Запуск регулярного подкаста НИУ ВШЭ о странах Азии", "Студенческий клуб \\"Говорим об Эфиопии\\"", "Визовое сопровождение иностранных студентов", "Международный мониторинг ответственного бизнеса", "Школа будущего международника: десятый сезон (2025/2026)", "Внешняя политика России в отношении стран Латинской Америки в XXI веке", "Ближневосточный клуб НИУ ВШЭ: платформа для изучения региона и развития академического и карьерного потенциала студентов-ближневосточников 2025/26", "Японский клуб «Мусуби» НИУ ВШЭ", "Практикум Совета молодых дипломатов", "Новости Ибероамерики-II", "Итальянский клуб ВШЭ", "Исследование правового обеспечения информационной безопасности стран БРИКС", "VII Международная конференция «Мировое боль

In [26]:
len(messages[0][1]["content"])

7586

In [27]:
answers = {}

In [28]:
for tag, message in tqdm.tqdm(list(zip(selected_tags, messages))):
    text = processor.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=2048)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

    json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if json_match:
        json_str = json_match.group(1)
    else:
        json_str = response

    answers.update({tag: json.loads(json_str)})

100%|██████████| 32/32 [05:32<00:00, 10.40s/it]


In [29]:
len(answers)

32

In [30]:
answers

{'international_relations': {'international_relations_specialist': 'A user deeply interested in international relations, with a strong focus on geopolitical analysis, regional studies (especially Eurasia, Asia, Middle East, and Latin America), BRICS dynamics, bilateral relations (Russia-China, Russia-Japan), international law/security issues, and the intersection of politics, economics, and media in global affairs. Highly engaged with academic and extracurricular international forums (Model UN, clubs, conferences).'},
 'sociology': {'sociology_and_cultural_studies_researcher': 'A user deeply interested in sociology, cultural studies, and intercultural communication. They are engaged in qualitative research (interviews, narrative analysis), media studies (film, podcasts, digital spaces), identity formation (ethnic, gender, political), social phenomena, and the intersection of culture, politics, and society. They value theoretical depth and practical applications in areas like social ent

In [31]:
with open("artificial_profiles.json", "w") as f:
    json.dump(answers, f)